In [53]:
import pandas as pd
from pathlib import Path


In [54]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/Data Preparation and Analysis")

In [56]:
# ...existing code...
from sklearn.model_selection import train_test_split

train_logs = pd.read_csv(data_path / "train_logs.csv")

# podział 80/20
X_train_scores, X_test_scores = train_test_split(
    train_logs,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# opcjonalnie: zapis, aby w kolejnych uruchomieniach móc je wczytywać z plików
X_train_scores.to_csv(data_path / "X_train_scores.csv", index=False)
X_test_scores.to_csv(data_path / "X_test_scores.csv", index=False)

print("Train:", X_train_scores.shape, "Test:", X_test_scores.shape)
# ...existing code...

Train: (6724718, 11) Test: (1681180, 11)


In [57]:
train_logs.head(10)

,id,event_id,down_time,up_time,action_time,activity,down_event,up_event,text_change,cursor_position,word_count
0,001519c8,1,4526,4557,31,Nonproduction,Leftclick,Leftclick,NoChange,0,0
1,001519c8,2,4558,4962,404,Nonproduction,Leftclick,Leftclick,NoChange,0,0
2,001519c8,3,106571,106571,0,Nonproduction,Shift,Shift,NoChange,0,0
3,001519c8,4,106686,106777,91,Input,q,q,q,1,1
4,001519c8,5,107196,107323,127,Input,q,q,q,2,1
5,001519c8,6,107296,107400,104,Input,q,q,q,3,1
6,001519c8,7,107469,107596,127,Input,q,q,q,4,1
7,001519c8,8,107659,107766,107,Input,q,q,q,5,1
8,001519c8,9,107743,107852,109,Input,q,q,q,6,1
9,001519c8,10,107840,107978,138,Input,Space,Space,,7,1


In [58]:
chars_per_essay = train_logs[train_logs['activity'] == 'Input'].groupby('id').agg(chars_per_essay=('event_id','nunique'))

In [59]:
chars_per_essay

,chars_per_essay
id,
001519c8,2010
0022f953,1938
0042269b,3515
0059420b,1304
0075873a,1942
...,...
ffb8c745,3588
ffbef7e5,2395
ffccd6fd,2849


In [60]:
grouped_train_logs = train_logs.groupby("id").agg(
    words_per_essay=("word_count", "last"),
    max_up_time=("up_time", "max"),
    min_down_time=("down_time", "min"),
    event_per_essay=("event_id", "nunique")
)

In [61]:
grouped_train_logs

,words_per_essay,max_up_time,min_down_time,event_per_essay
id,,,,
001519c8,255,1801969,4526,2557
0022f953,320,1788969,30623,2454
0042269b,404,1771669,4441,4136
0059420b,206,1404469,41395,1556
0075873a,252,1662472,78470,2531
...,...,...,...,...
ffb8c745,273,1791649,22467,4739
ffbef7e5,438,1799174,21732,2604
ffccd6fd,201,1959363,23482,3063


In [62]:
grouped_train_logs ['writing_duration'] = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

In [63]:
grouped_train_logs = grouped_train_logs.merge(chars_per_essay, left_index=True, right_index=True)

In [64]:
grouped_train_logs = grouped_train_logs['max_up_time'] - grouped_train_logs['min_down_time']

In [ ]:
grouped_train_logs ['chars_per_minute'] = 1000 * 60 * grouped_train_logs['chars_per_essay'] / grouped_train_logs['writing_duration']

KeyError: 'chars_per_essay'

### Pauses

In [66]:
train_logs.sort_values(by=['id', 'down_time'], inplace=True)
train_logs['pause_time'] = (train_logs.groupby('id')['down_time'].shift(0) - train_logs.groupby('id')['up_time'].shift(1)).fillna(0)

In [ ]:
train_logs

In [ ]:
train_logs['is pause'] = train_logs['pause_time'] > 2000

In [ ]:
pause_features = train_logs.groupby('id').agg(
    total_pauses=('pause_time', 'sum'),
    total_pause_time=('pause_time', 'sum')
)

In [ ]:
pause_features

In [ ]:
grouped_train_logs = grouped_train_logs.merge(pause_features, left_index=True, right_index=True)

In [ ]:
grouped_train_logs

In [ ]:
grouped_train_logs['perc_pause_time'] = grouped_train_logs['total_pause_time'] / grouped_train_logs['writing_duration']

In [ ]:
grouped_train_logs['pauses_per_words'] = grouped_train_logs['total_pauses'] / grouped_train_logs['words_per_essay']

### Aktywności

In [ ]:
train_logs['activity'].value_counts()